# Notebook 3B — AWS Glue PySpark ETL

**Purpose:** show how the same notebook logic from Use Case 2 becomes a repeatable **AWS Glue PySpark** pipeline.

## Where this notebook should be run
- **AWS Glue Studio notebook**, or
- **AWS Glue interactive session**


## Dependencies and AWS context

### Glue / Spark dependencies
- **SparkSession** — entry point for distributed processing
- **pyspark.sql.functions** — used for date parsing, null handling, derived columns, and aggregations

### One-time setup before the live Glue demo
Update the **bucket** and optional **prefix** in the config cell below.
Once updated, the notebook points to the exact S3 locations created by Use Case 1 and Use Case 2.


In [2]:
import sys
!{sys.executable} -m pip install pyspark
from pyspark.sql import SparkSession
import os

# -------------------------------
# Config (Local File System Setup)
# -------------------------------
AWS_REGION = os.getenv('AWS_REGION', 'ap-south-1')

# Local paths replacing S3 URIs
INPUT_PATH = "./retail_exploration_ready.csv"
OUT_DAILY = "./daily_country_revenue_glue/"
OUT_MONTHLY = "./monthly_category_revenue_glue/"

# -------------------------------
# Spark Session (Configured for Local Execution)
# -------------------------------
spark = SparkSession.builder \
    .appName('Local-UseCase2-ETL') \
    .master('local[*]') \
    .getOrCreate()

print('Local input path   :', INPUT_PATH)
print('Local daily output :', OUT_DAILY)
print('Local monthly output:', OUT_MONTHLY)

     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/4

## Step 1 — Extract from S3

Glue reads the prepared file from a shared S3 location so every worker can access the same input.


In [3]:
raw_df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')
         .csv(INPUT_PATH)
)

print('Raw row count:', raw_df.count())
raw_df.show(5, truncate=False)


Raw row count: 500
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889     |Belgium       |2011-02-01 11:08:00|
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943     |Germany       |2011-01-28 11:32:00|
|536366   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|6       |02/05/2011 08:48|5.8      |18065     |Netherlands   |2011-02-05 08:48:00|
|536366   |22752    |SET 7 BABUSHKA NESTING BOXES     |4       |01/13/2011 13:54|7.55     |14512     |United Kingdom|2011-01-13 13:54:00|
|536366   |2173

## Step 2 — Transform and clean with PySpark

This cell mirrors the logic from Notebook 2 and Notebook 3, but in a distributed engine.


In [5]:
from pyspark.sql import functions as F

clean_df = (
    raw_df
      .withColumn('InvoiceDateTs', F.to_timestamp(F.col('InvoiceDate'), 'MM/dd/yyyy HH:mm'))
      .withColumn('Description', F.coalesce(F.col('Description'), F.lit('UNKNOWN_ITEM')))
      .withColumn('CustomerID', F.coalesce(F.col('CustomerID').cast('string'), F.lit('UNKNOWN_CUSTOMER')))
      .filter(F.col('InvoiceDateTs').isNotNull())
      .filter(F.col('UnitPrice') > 0)
      .withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
      .withColumn('IsReturn', F.col('Quantity') < 0)
      .withColumn('TransactionDate', F.to_date('InvoiceDateTs'))
      .withColumn('Month', F.date_format('InvoiceDateTs', 'yyyy-MM'))
      .withColumn('Category', F.split(F.col('Description'), ' ').getItem(0))
)

clean_df.show(5, truncate=False)

+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |InvoiceDateTs      |Revenue|IsReturn|TransactionDate|Month  |Category|
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889     |Belgium       |2011-02-01 11:08:00|2011-02-01 11:08:00|32.94  |false   |2011-02-01     |2011-02|WHITE   |
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943     |Germany       |2011-01-28 11:32:00|2011-01-28 11:32:00|8.44   |fal

## Step 3 — Aggregate into ETL outputs

This is the distributed equivalent of the pandas `groupby` logic.


In [6]:
daily_country_revenue = (
    clean_df.groupBy('TransactionDate', 'Country')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

monthly_category_revenue = (
    clean_df.groupBy('Month', 'Category')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

daily_country_revenue.show(10, truncate=False)
monthly_category_revenue.show(10, truncate=False)


+---------------+-----------+-------+
|TransactionDate|Country    |Revenue|
+---------------+-----------+-------+
|2011-02-23     |France     |9.52   |
|2011-02-20     |Belgium    |56.64  |
|2011-02-07     |Spain      |4.03   |
|2011-03-14     |Belgium    |7.12   |
|2011-02-08     |Spain      |32.76  |
|2011-03-30     |Germany    |72.36  |
|2011-02-03     |France     |85.71  |
|2011-01-17     |Netherlands|59.64  |
|2011-02-06     |France     |33.16  |
|2011-03-09     |Spain      |9.04   |
+---------------+-----------+-------+
only showing top 10 rows
+-------+------------+-------+
|Month  |Category    |Revenue|
+-------+------------+-------+
|2011-01|CREAM       |314.82 |
|2011-03|SET         |659.26 |
|2011-03|HAND        |1132.78|
|2011-02|HAND        |1298.91|
|2011-01|UNKNOWN_ITEM|232.68 |
|2011-03|CREAM       |377.02 |
|2011-02|GLASS       |619.36 |
|2011-02|ASSORTED    |472.16 |
|2011-03|UNKNOWN_ITEM|64.56  |
|2011-01|WHITE       |688.86 |
+-------+------------+-------+
only show

## Step 4 — Write outputs back to S3

Glue writes folders to S3 because Spark outputs distributed files rather than a single local CSV.


In [9]:
import os
from pathlib import Path
import urllib.request
import tempfile

# 1. Create a local winutils directory structure
hadoop_home = Path(tempfile.gettempdir()) / "hadoop"
bin_dir = hadoop_home / "bin"
bin_dir.mkdir(parents=True, exist_ok=True)

# 2. Download the required winutils.exe for Hadoop on Windows if it doesn't exist
winutils_path = bin_dir / "winutils.exe"
if not winutils_path.exists():
    print("📥 Downloading winutils.exe for Hadoop compatibility on Windows...")
    url = "https://github.com/steveloughran/winutils/raw/master/hadoop-3.0.0/bin/winutils.exe"
    urllib.request.urlretrieve(url, winutils_path)

# 3. Point Hadoop home environment variable and JVM system property to it
os.environ["HADOOP_HOME"] = str(hadoop_home.absolute())

print("✅ Hadoop environment configured successfully!")

📥 Downloading winutils.exe for Hadoop compatibility on Windows...
✅ Hadoop environment configured successfully!


In [11]:
import datetime, pytz; 
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-08-31 22:59:48
